In [2]:
import pylablib as pll
pll.par["devices/dlls/andor_sdk2"] = r"D:\instrument_control_v3_1\andor dll"
from pylablib.devices import Andor
import numpy as np
import matplotlib.pyplot as plt
import time
%matplotlib qt


In [6]:
## control sr500i
pll.par["devices/dlls/andor_shamrock"] = r"D:\instrument_control_v3_1\andor dll"
spec = Andor.ShamrockSpectrograph()

OSError: can't import library atspectrograph.dll or ShamrockCIF64.dll or ShamrockCIF.dll
The library is automatically supplied with Andor SDK2 software or micromanager plugin
If you already have it, specify its path as pylablib.par['devices/dlls/andor_shamrock']='path/to/dll/'

In [3]:
Andor.get_cameras_number_SDK2()

2

In [10]:
cam_si = Andor.AndorSDK2Camera(idx=0)
cam_ingaas = Andor.AndorSDK2Camera(idx=1)
# cam_si.close()
# cam_ingaas.close()

In [21]:
## set cooler on
cam_si.set_cooler(on=True)
cam_ingaas.set_cooler(on=True)

True

In [237]:
## MUST close cam after process
cam_si.close()
cam_ingaas.close()

In [210]:
cam_ingaas.get_temperature()

-55.882999420166016

In [209]:
cam_si.get_temperature()

-63.67100143432617

In [27]:
cam_si.get_shutter_parameters()

('closed', 0, 0, 0)

In [28]:
cam_si.setup_shutter('auto')

('auto', 0, 0, 0)

In [30]:
cam_ingaas.setup_shutter('auto')

('auto', 0, 0, 0)

In [32]:
cam_ingaas.set_fan_mode('full')

'full'

### set acquision parameters

In [235]:
def get_one_spectra(cam = cam, exposure_time = 1, num_acc = 1, plot = False):
    cam.set_exposure(exposure_time)
    cam.setup_accum_mode(num_acc=num_acc,cycle_time_acc=1)
    cam.setup_acquisition(mode='snap', nframes=num_acc)
    cam.start_acquisition()
    # 🔁 Wait for acquisition to complete
    while True:
        status = cam.get_status()
        if status.lower() == 'idle':
            break
        time.sleep(0.1)  # sleep briefly to avoid hogging CPU
    frames,_= cam._read_frames((0, 1))
    frame = frames[0]  
    if plot == True:
        x_pixies = np.linspace(1,512,512)
        plt.plot(x_pixies, frame[0])
        plt.title(f"exp = {exposure_time} ")
        plt.xlabel("Wavelength (nm)")
        plt.ylabel("Intensity")
        plt.grid(True)
        plt.show()
    cam.clear_acquisition()  
    return frame

In [236]:
data = get_one_spectra(cam=cam_ingaas, exposure_time = 1, num_acc = 1, plot = True)

In [213]:
cam_ingaas.set_exposure(10)

10.000000953674316